
Для ізоляції залежностей проєкту та забезпечення відтворюваності результатів використовується віртуальне середовище Python.
1. Створення середовища: python -m venv venv.
2. Активація та встановлення залежностей: pip install -r requirements.txt
3. Підключаємо бібліотеки:

In [3]:
import piplite
await piplite.install(['pandas'])
import pandas as pd
from pyodide.http import open_url
import os
from datetime import datetime

Для кожної з адміністративних одиниць України завантажити (urllib)
тестові структуровані файли, що містять значення VHI-індексу.
При зберіганні файлу, до його імені потрібно додати дату та час 
завантаження. Передбачити повторні запуски скрипту,
реалізувати механізм запобігання повторного
довантаження та колізії даних;


In [4]:
def download_vhi_data():
    if not os.path.exists('data'):
        os.makedirs('data')
        
    for province_id in range(1, 26):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
 
        existing_files = [f for f in os.listdir('data') if f.startswith(f'vhi_id_{province_id}_')]
        if existing_files:
            print(f"Data for province {province_id} is already loaded.")
            continue
            
        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"data/vhi_id_{province_id}_{now}.csv"
        
        try:
            response = open_url(url)
            content = response.read()

            with open(filename, 'w') as f:
                f.write(content)
            print(f"Loaded: {filename}")
        except Exception as e:
            print(f"Error for province {province_id}: {e}")

download_vhi_data()

Data for province 1 is already loaded.
Data for province 2 is already loaded.
Data for province 3 is already loaded.
Data for province 4 is already loaded.
Data for province 5 is already loaded.
Data for province 6 is already loaded.
Data for province 7 is already loaded.
Data for province 8 is already loaded.
Data for province 9 is already loaded.
Data for province 10 is already loaded.
Data for province 11 is already loaded.
Data for province 12 is already loaded.
Data for province 13 is already loaded.
Data for province 14 is already loaded.
Data for province 15 is already loaded.
Data for province 16 is already loaded.
Data for province 17 is already loaded.
Data for province 18 is already loaded.
Data for province 19 is already loaded.
Data for province 20 is already loaded.
Data for province 21 is already loaded.
Data for province 22 is already loaded.
Data for province 23 is already loaded.
Data for province 24 is already loaded.
Data for province 25 is already loaded.


Зчитати завантажені текстові файли у pandas dataframe. 
дійснити data cleaning: прибрати зайві стовпці, 
заповнити пропуски, видалити зайвий текст тощо.
Додати стовпчики з назвою та індексом області

In [7]:
names = {1: "Vinnytsia", 2: "Volyn", 3: "Dnipropetrovsk",
         4: "Donetsk", 5: "Zhytomyr", 6: "Zakarpattia",
         7: "Zaporizhzhia", 8: "Ivano-Frankivsk", 9: "Kyiv",
         10: "Kirovohrad", 11: "Luhansk", 12: "Lviv",
         13: "Mykolaiv", 14: "Odesa", 15: "Poltava",
         16: "Rivne", 17: "Sumy", 18: "Ternopil", 19: "Kharkiv",
         20: "Kherson", 21: "Khmelnytskyi", 22: "Cherkasy", 
         23: "Chernivtsi", 24: "Chernihiv", 25: "Crimea"}

def create_clean_dataframe(folder_path):
    all_data = []
    files = [f for f in os.listdir(folder_path) if f.startswith('vhi_id_') and f.endswith('.csv')]
    
    for file in files:
        parts = file.split('_')
        noaa_id = int(parts[2])
        path = os.path.join(folder_path, file)

        df = pd.read_csv(path, index_col=None, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'extra'])
 
        df = df.drop(columns=['extra'])
        df = df.dropna()

        df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '', regex=False).str.replace('</pre></tt>', '', regex=False)
        df = df[df['VHI'] != -1]
        df['Area_ID'] = id_map[noaa_id]
        
        all_data.append(df)
        
    if not all_data:
        return pd.DataFrame() 
        
    return pd.concat(all_data, ignore_index=True)

df = create_clean_dataframe('data')

df['Region'] = df['Area_ID'].map(names)
df['Year'] = df['Year'].astype(int)
df['VHI'] = df['VHI'].astype(float)

print(df.head())

   Year  Week    SMN     SMT    VCI    TCI    VHI  Area_ID        Region
0  1982   1.0  0.059  258.24  51.11  48.78  49.95       21  Khmelnytskyi
1  1982   2.0  0.063  261.53  55.89  38.20  47.04       21  Khmelnytskyi
2  1982   3.0  0.063  263.45  57.30  32.69  44.99       21  Khmelnytskyi
3  1982   4.0  0.061  265.10  53.96  28.62  41.29       21  Khmelnytskyi
4  1982   5.0  0.058  266.42  46.87  28.57  37.72       21  Khmelnytskyi


Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [10]:
id_map = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 13, 13: 14, 14: 15, 15: 16, 16: 17, 17: 18, 18: 6, 19: 1,
    20: 2, 21: 7, 22: 5, 23: 10, 24: 11, 25: 12
}

df['Area_Name'] = df['Area_ID'].map(names)

Викличему функцію, відсортовану по айді з унікальними регіонами і перевіримо:

In [13]:
unique_regions_sample = df.sort_values(by='Area_ID').drop_duplicates(subset=['Area_ID']).head(12)

display(unique_regions_sample[['Area_ID', 'Area_Name', 'Year', 'VHI']])

,Area_ID,Area_Name,Year,VHI
21858,1,Vinnytsia,2024,42.49
26230,2,Volyn,2024,48.89
44450,3,Dnipropetrovsk,1996,46.61
48091,4,Donetsk,2024,32.09
29871,5,Zhytomyr,2010,48.72
19671,6,Zakarpattia,2024,38.54
26958,7,Zaporizhzhia,1996,59.04
49545,8,Ivano-Frankivsk,2010,44.13
4100,9,Kyiv,2019,26.04
31695,10,Kirovohrad,2003,48.40


Реалізувати процедури для формування вибірок наступного виду:
    Ряд VHI для області за вказаний рік;
    Ряд VHI за вказаний діапазон років для вказаних областей;
    Пошук екстремумів (min та max) для вказаних областей та років,  
    середнього, медіани;


In [19]:
def get_vhi_by_area_and_year(df, area_id, year):
    result = df[(df['Area_ID'] == area_id) & (df['Year'] == year)]
    
    return result[['Week', 'VHI']].reset_index(drop=True)


def get_vhi_multiple_areas_range(df, area_ids, start_year, end_year):
    filter_condition = (df['Area_ID'].isin(area_ids)) & (df['Year'].between(start_year, end_year))
    result = df[filter_condition]
    
    return result[['Area_ID', 'Year', 'Week', 'VHI']].sort_values(by=['Area_ID', 'Year', 'Week'])

def get_vhi_statistics(df, area_ids, start_year, end_year):
    subset = df[(df['Area_ID'].isin(area_ids)) & (df['Year'].between(start_year, end_year))]
    stats = {
        "Min VHI": subset['VHI'].min(),
        "Max VHI": subset['VHI'].max(),
        "Mean VHI": subset['VHI'].mean(),
        "Median VHI": subset['VHI'].median()
    }
    print(f" Statistics for Areas {area_ids} ({start_year}-{end_year})")
    for key, value in stats.items():
        print(f"{key}: {value:.2f}")
    
    return stats

Перевірка по вибірці за рік(для Вінниці 2023):

In [17]:
print("VHI Series for Vinnytsia (2023):")
vhi_2023 = get_vhi_by_area_and_year(df, 1, 2023)
print(vhi_2023.head(10))

VHI Series for Vinnytsia (2023):
   Week    VHI
0   1.0  31.96
1   2.0  33.95
2   3.0  36.70
3   4.0  39.71
4   5.0  43.16
5   6.0  45.60
6   7.0  47.78
7   8.0  51.66
8   9.0  53.93
9  10.0  53.50


Перевірка по вибірці за  вказаний діапазон років для вказаних областей(Волинь та Дніпро 2020-2021):

In [18]:
print("VHI Range for Volyn and Dnipro (2020-2021):")
vhi_range = get_vhi_multiple_areas_range(df, [2, 3], 2020, 2021)
print(vhi_range)

VHI Range for Volyn and Dnipro (2020-2021):
       Area_ID  Year  Week    VHI
25972        2  2020   1.0  36.76
25973        2  2020   2.0  38.55
25974        2  2020   3.0  39.57
25975        2  2020   4.0  40.44
25976        2  2020   5.0  41.22
...        ...   ...   ...    ...
45745        3  2021  48.0  42.21
45746        3  2021  49.0  44.24
45747        3  2021  50.0  45.90
45748        3  2021  51.0  48.01
45749        3  2021  52.0  49.58

[208 rows x 4 columns]


Перевірка пошуку екстремумів для вказаних областей та років, середнього, медіани(для Львову і Вінниці 2015-2024):


In [21]:
get_vhi_statistics(df, [1, 12], 2015, 2024)

 Statistics for Areas [1, 12] (2015-2024)
Min VHI: 31.80
Max VHI: 70.24
Mean VHI: 50.26
Median VHI: 49.83


{'Min VHI': np.float64(31.8),
 'Max VHI': np.float64(70.24),
 'Mean VHI': np.float64(50.255192307692305),
 'Median VHI': np.float64(49.825)}